# 4-1 プロジェクトの全体像

ここまでで Python の基本・pandas・matplotlib を一通りやりました。第 4 章では、**それを全部つないで「実務の自動化スクリプト」を組み立てます**。

## シナリオ

**あなたは食品流通会社の本社事務。** 全国 47 都道府県に支店があり、**月初に各支店から前月の売上 Excel** が送られてきます。

- 47 ファイル × 各 1,000 行 ≒ 全社で **約 47,000 行**
- これを 1 つにまとめて、**全社月次レポート** を作るのが今月の仕事
- これまでは手動で 47 ファイルを開いて、Excel に貼り付けて、ピボットテーブルを作って…**ほぼ半日仕事**
- Python なら **セル数個 + 数秒** で終わる

本章のゴールは「**毎月セルを 1 つ実行するだけで、レポート Excel が完成する**」状態を作ることです。

## 完成イメージ

最終的に `output/report_2026-01.xlsx` という Excel ファイルが出力されます。中身はこんな構成:

| シート | 中身 |
|---|---|
| 概要 | 全社の売上合計・取引件数・営業日・グラフ |
| 地域別 | 地域ブロック（北海道/東北/関東…）別の売上・利益 |
| 都道府県別 | 47 都道府県の売上ランキング |
| 商品別 | 6 商品の売上・利益・カテゴリ別集計 |
| 明細 | 全 47,000 行の元データ（フィルタ用） |

さらに `output/charts/` フォルダに、各種グラフを **PNG 画像** で保存しておきます（レポートやメール本文に貼れる形）。

## 章ごとの担当

やることが多いので、**章ごとに 1 工程ずつ** 進めていきます。

| 章 | やること | 思い出す章 |
|---|---|---|
| **4-2** | 47 ファイルを **一括で読み込み**、1 つの DataFrame に | 2-2 (Excel 読込) |
| **4-3** | 商品マスタ + 支店マスタを **結合** し、地域・カテゴリで集計 | 2-3 / 2-4 (集計・加工) |
| **4-4** | 集計結果を **グラフ** にする、PNG で保存 | 3章 全部 |
| **4-5** | 複数シートの **Excel レポート** を書き出す | 2-5 (Excel 書込) |
| **4-6** | 全部を **1 つの関数** にまとめ、来月も使える形に | 1-5 (関数) |
| **4-7** | 同じ処理を **VBA** で書いて、**速度を比較** | (おまけ) |

1〜3 章でやったことを **そのまま部品として使う** だけです。新しい難しい概念は出てきません。

## 使うデータ

サンプルデータは `data/capstone/` 配下に既に用意してあります。

```
data/capstone/
├── sales_2026-01/                                ← 47 ファイル
│   ├── sales_2026-01_01_hokkaido.xlsx           ← 北海道 (1,000行)
│   ├── sales_2026-01_02_aomori.xlsx             ← 青森
│   ├── sales_2026-01_03_iwate.xlsx              ← 岩手
│   ├── …
│   └── sales_2026-01_47_okinawa.xlsx            ← 沖縄
├── master_products.xlsx                          ← 商品マスタ (6 行)
└── master_branches.xlsx                          ← 支店マスタ (47 行)
```

**ファイル名のルール**: `sales_<年月>_<県コード>_<英字県名>.xlsx`

- `<年月>`: `2026-01`
- `<県コード>`: `01` (北海道) 〜 `47` (沖縄)
- `<英字県名>`: `hokkaido`, `aomori`, ...

## 準備 — Colab かローカルかで最初のセルだけ違う

第 4 章は、**Colab で進める人** も **ローカル (uv) で進める人** も、どちらでも進められるようになっています。違うのは **次のセルの最初の数行だけ** です。

以下のセルは **両方のパターンを書いていて、自分の環境に合う方のコメントを外して使う** スタイルです。

In [ ]:
# === Colab で進める人 ===
# 次の 3 行のコメントを外して実行してください
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカル (uv + Jupyter Lab) で進める人 ===
# 何もしなくて OK。今いるフォルダ (python-data-basics-local/) を基準にします
BASE = "."

# 第 4 章で使うフォルダ
DATA = f"{BASE}/data/capstone"
SALES_DIR = f"{DATA}/sales_2026-01"
OUT = f"{BASE}/output"

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/charts", exist_ok=True)

print(f"DATA      = {DATA}")
print(f"SALES_DIR = {SALES_DIR}")
print(f"OUT       = {OUT}")

## サンプルデータを覗いてみる

**まず 1 ファイル開いて、中身がどうなっているか** を確認しましょう。

In [ ]:
import pandas as pd

# 1 ファイル(東京)を開いてみる
df_tokyo = pd.read_excel(f"{SALES_DIR}/sales_2026-01_13_tokyo.xlsx", sheet_name="売上")
print(f"行数: {len(df_tokyo):,}")
df_tokyo.head()

**列の意味**:

| 列 | 意味 |
|---|---|
| `date` | 売上日 |
| `product_code` | 商品コード (P001〜P006) |
| `product_name` | 商品名 |
| `quantity` | 数量 |
| `unit_price` | 単価 (円) |
| `amount` | 売上金額 = `quantity × unit_price` (円) |

**この時点では「都道府県」「カテゴリ」「原価」が無い** ことに注目してください。それは **マスタ** に入っていて、4-3 で結合します。

In [ ]:
# 商品マスタ — 商品コード ↔ カテゴリ・原価
master_products = pd.read_excel(f"{DATA}/master_products.xlsx")
master_products

In [ ]:
# 支店マスタ — 都道府県コード ↔ 県名・地域ブロック
master_branches = pd.read_excel(f"{DATA}/master_branches.xlsx")
print(f"支店数: {len(master_branches)}")
master_branches.head(10)

In [ ]:
# 地域ブロックの内訳
master_branches.groupby("region").size().sort_values(ascending=False)

## 47 ファイルあることを確認

`sales_2026-01/` フォルダの中に **本当に 47 ファイルあるか** 数えておきましょう。

In [ ]:
from pathlib import Path

files = sorted(Path(SALES_DIR).glob("*.xlsx"))
print(f"ファイル数: {len(files)}")
for f in files[:3]:
    print(f"  {f.name}")
print("  ...")
for f in files[-3:]:
    print(f"  {f.name}")

47 ファイル見えていれば準備 OK。次の 4-2 で、**これを 1 つの DataFrame に束ねる** ことをやります。

## VBA でやろうとするとどうなる？

同じことを VBA でやろうとすると、おおよそ次のような流れになります:

```vba
Sub AggregateMonthlySales()
    Dim folder As String, file As String
    folder = ThisWorkbook.Path & "\sales_2026-01\"
    file = Dir(folder & "sales_*.xlsx")
    Do While file <> ""
        Dim wb As Workbook
        Set wb = Workbooks.Open(folder & file, ReadOnly:=True)   ' ← Excel が起動!
        ' …シートを読み込んで、行を一つずつループして、Dictionary に集計…
        wb.Close SaveChanges:=False                              ' ← Excel を閉じる
        file = Dir
    Loop
End Sub
```

ポイントは **`Workbooks.Open`**。これは「Excel というアプリでファイルを開く」操作です。

- 1 ファイルあたり Excel のプロセスを起動 → 描画 → 計算エンジン読込 → 全行を 1 つずつスキャン → 閉じる
- これを **47 回繰り返す** ので、どうしてもファイル数に比例して時間がかかる

一方の pandas は **Excel ファイル (xlsx は実は ZIP) を Python が直接パース** します。Excel アプリを使わない。だから速い。

実際にどれだけ違うかは、**章のラスト 4-7** で実測して比較します。お楽しみに。

## まとめ

- 第 4 章のゴールは「**月初にセル 1 つ実行 → 全社月次レポートが完成**」する状態を作ること
- 題材は **47 都道府県の支店から届く 1 ヶ月分の売上ファイル**（合計約 47,000 行）
- 売上は **支店ごとに 1 ファイル**、商品マスタ・支店マスタは別 Excel に分かれている
- 章ごとに「読込 → 結合 → 集計 → グラフ → Excel 書込 → 関数化」と 1 工程ずつ進める

次の **4-2** では、47 ファイルを **`glob` と `for` ループ** で一気に読み込んで、`pd.concat` で 1 つの DataFrame に束ねます。VBA だと一番手間がかかる部分が、たった 4〜5 行で終わります。